Cleaning

In [ ]:
#0　基本設定

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

%matplotlib inline
%config Inlinebackend.figure_format='retina'
sns.set(style='whitegrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',100)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
'''
クリーニング手順メモ
1 読み込み → 即ダウンキャスト、オブジェクト型の変換
2 各テーブル参照、マージ検討
3 テーブルマージ及び再ダウンキャスト
4 金額列だけfloat32→float64（桁あふれ対策）
5 モデル(LightGBM / CatBoost)試走
6 リーク検討
7 モデル再試走(リークがなければ省略)
※手順8中止　後のEDA等で使用するため　8 両モデルでimportance=0の列を削除
9 モデル再試走　影響度確認
10 parquet（zstd）出力
'''

'\nクリーニング手順メモ\n1 読み込み → 即ダウンキャスト、オブジェクト型の変換\n2 各テーブル参照、マージ検討\n3 テーブルマージ及び再ダウンキャスト\n4 金額列だけfloat32→float64（桁あふれ対策）\n5 モデル(LightGBM / CatBoost)試走\n6 リーク検討\n7 モデル再試走(リークがなければ省略)\n8 両モデルでimportance=0の列を削除\nなお、本データは削除せずFE後に試走し、不要カラムを削除\n9 モデル再試走\u3000影響度確認\n10 parquet（zstd）出力\n'

In [ ]:
#1-1　ダウンキャスト関数（int,float）

total_mem = 0
total_mem_r = 0

def reduce(df):
    global total_mem,total_mem_r

    """ 数値データの型を最適化してメモリを節約する関数 """
    start_mem=df.memory_usage().sum()/1024**2

    numeric_cols = df.select_dtypes(include=['number']).columns

    for col in numeric_cols:
        col_type = df[col].dtype
        cmin = df[col].min()
        cmax = df[col].max()

        if str(col_type)[:3]=='int':
            if   cmin > np.iinfo(np.int8).min and cmax < np.iinfo(np.int8).max:
              df[col]=df[col].astype(np.int8)
            elif cmin > np.iinfo(np.int16).min and cmax < np.iinfo(np.int16).max:
              df[col]=df[col].astype(np.int16)
            elif cmin > np.iinfo(np.int32).min and cmax < np.iinfo(np.int32).max:
              df[col]=df[col].astype(np.int32)
            else:
              df[col]=df[col].astype(np.int64)
        else:
            #if   cmin > np.finfo(np.float16).min and cmax < np.finfo(np.float16).max:
              #df[col]=df[col].astype(np.float16)
            #elif       今回はfloat16型不使用
            if   cmin > np.finfo(np.float32).min and cmax < np.finfo(np.float32).max:
              df[col]=df[col].astype(np.float32)
            else:
              df[col]=df[col].astype(np.float64)

    end_mem=df.memory_usage().sum()/1024**2
    print(f'メモリ使用量: {start_mem:.2f}MB -> {end_mem:.2f}MB | ',
          f'削減: {(start_mem - end_mem) / start_mem:.2f}MB {100*(start_mem - end_mem) / start_mem:.1f}%')
    total_mem +=start_mem
    total_mem_r +=end_mem
    return df

In [ ]:
#1-2　csv読込→ダウンキャスト
application_train=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/application_train.csv')
application_train=reduce(application_train) #メインテーブル train
application_test=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/application_test.csv')
application_test=reduce(application_test) #メインテーブル test

bureau=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/bureau.csv')
bureau=reduce(bureau) #結合先：メイン、キー：SK_ID_CURR

bureau_balance=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/bureau_balance.csv')
bureau_balance=reduce(bureau_balance) #結合先：bureau、キー：SK_ID_BUREAU

previous_application=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/previous_application.csv')
previous_application=reduce(previous_application) #結合先：メイン、キー：SK_ID_CURR

POS_CASH_balance=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/POS_CASH_balance.csv')
POS_CASH_balance=reduce(POS_CASH_balance) #結合先：previous_application、キー：SK_ID_PREV

instalments_payments=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/installments_payments.csv')
instalments_payments=reduce(instalments_payments) #結合先：previous_application、キー：SK_ID_PREV

credit_card_balance=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/原データ/credit_card_balance.csv')
credit_card_balance=reduce(credit_card_balance) #結合先：previous_application、キー：SK_ID_PREV


total_reduce = total_mem - total_mem_r
total_ratio = 100 * (total_reduce/total_mem)
print('-'*20)
print(f'合計メモリ使用量：{total_mem:.2f}MB -> {total_mem_r:.2f}MB | ',
      f'合計削減: {(total_reduce):.2f}MB {(total_ratio):.2f}%')

メモリ使用量: 286.23MB -> 128.16MB |  削減: 0.55MB 55.2%
メモリ使用量: 45.00MB -> 20.27MB |  削減: 0.55MB 55.0%
メモリ使用量: 222.62MB -> 119.49MB |  削減: 0.46MB 46.3%
メモリ使用量: 624.85MB -> 338.46MB |  削減: 0.46MB 45.8%
メモリ使用量: 471.48MB -> 324.94MB |  削減: 0.31MB 31.1%
メモリ使用量: 610.43MB -> 276.60MB |  削減: 0.55MB 54.7%
メモリ使用量: 830.41MB -> 389.25MB |  削減: 0.53MB 53.1%
メモリ使用量: 673.88MB -> 318.63MB |  削減: 0.53MB 52.7%
--------------------
合計メモリ使用量：3764.90MB -> 1915.80MB |  合計削減: 1849.09MB 49.11%


In [ ]:
'''
テーブル構造メモ
application_train/test（メイン）
  └─ SK_ID_CURR ─ bureau（他社借入履歴）
               │    └─ SK_ID_BUREAU ─ bureau_balance（月次残高）
               └─ previous_application（過去申請）
                     └─ SK_ID_PREV ─ POS_CASH_balance（月次残高）
                                　   ─ instalments_payments（返済履歴）
                             　      ─ credit_card_balance（CC残高）
'''

'\nテーブル構造メモ\napplication_train/test（メイン）\n  └─ SK_ID_CURR ─ bureau（他社借入履歴）\n               │    └─ SK_ID_BUREAU ─ bureau_balance（月次残高）\n               └─ previous_application（過去申請）\n                     └─ SK_ID_PREV ─ POS_CASH_balance（月次残高）\n                                \u3000   ─ instalments_payments（返済履歴）\n                             \u3000      ─ credit_card_balance（CC残高）\n'

In [ ]:
#1-3　各テーブル確認　(データ型の確認)

tables = {
    "application_train": application_train,
    "application_test": application_test,
    "bureau": bureau,
    "bureau_balance": bureau_balance,
    "previous_application": previous_application,
    "POS_CASH_balance": POS_CASH_balance,
    "instalments_payments": instalments_payments,
    "credit_card_balance": credit_card_balance
}

summary_data = []

for name, df in tables.items():
    type_counts = df.dtypes.value_counts()
    info = {
        "Table": name,
        "Total Columns": len(df.columns),
        "Rows": len(df)
    }
    for dtype, count in type_counts.items():
        info[str(dtype)] = info.get(str(dtype), 0) + count

    summary_data.append(info)

summary_df = pd.DataFrame(summary_data).fillna(0).astype({col: 'int' for col in summary_data[0].keys() if col != 'Table'})
summary_df

,Table,Total Columns,Rows,float32,int8,object,int32,int16
0,application_train,122,307511,65,37,16,2,2
1,application_test,121,48744,65,36,16,2,2
2,bureau,17,1716428,8,1,3,3,2
3,bureau_balance,3,27299925,0,1,1,1,0
4,previous_application,37,1670214,15,2,16,3,1
5,POS_CASH_balance,8,10001358,2,1,1,2,2
6,instalments_payments,8,13605401,5,0,0,2,1
7,credit_card_balance,23,3840312,15,1,1,3,3


In [ ]:
#1-4　データ型変換(オブジェクト→カテゴリ型)

for df in tables.values():
    obj_cols = df.select_dtypes(include='object').columns
    for col in obj_cols:
        df[col] = df[col].astype('category')

for name, df in tables.items():
    print(f"{name:25} | Object: {len(df.select_dtypes('object').columns)} | Category: {len(df.select_dtypes('category').columns)}")

application_train         | Object: 0 | Category: 16
application_test          | Object: 0 | Category: 16
bureau                    | Object: 0 | Category: 3
bureau_balance            | Object: 0 | Category: 1
previous_application      | Object: 0 | Category: 16
POS_CASH_balance          | Object: 0 | Category: 1
instalments_payments      | Object: 0 | Category: 0
credit_card_balance       | Object: 0 | Category: 1


In [ ]:
#2-1　各テーブル参照、マージ検討  (bureau及びbureau_balance　キー：SK_ID_BUREAU)

for name in ["bureau", "bureau_balance"]:
    df = tables[name]
    print(f"\n" + "="*60)
    print(f"【Table: {name}】 Unique Value Analysis")
    print(f"="*60)

    for col in df.columns:
        unique_count = df[col].nunique()
        print(f"\n[Column: {col}]")
        print(f"  - Unique values: {unique_count}")

        # カテゴリ型、オブジェクト型、または種類が少ない数値型（フラグ等）の内訳を表示
        if df[col].dtype.name in ['category', 'object'] or unique_count < 15:
            print("  - Value counts:")
            print(df[col].value_counts(dropna=False).head(10)) # 上位10件
        else:
            # 連続値（金額など）の場合は、代表的な統計量を添える
            print(f"  - Min: {df[col].min()} | Max: {df[col].max()} | Mean: {df[col].mean():.2f}")
    print("\n")

# bureau (他社借入履歴)
print(f"\n{'='*50}\n【Table: bureau】\n{'='*50}")
print(tables['bureau'].head())

# bureau_balance (他社借入の月次残高)
print(f"\n{'='*50}\n【Table: bureau_balance】\n{'='*50}")
print(tables['bureau_balance'].head())



【Table: bureau】 Unique Value Analysis

[Column: SK_ID_CURR]
  - Unique values: 305811
  - Min: 100001 | Max: 456255 | Mean: 278214.93

[Column: SK_ID_BUREAU]
  - Unique values: 1716428
  - Min: 5000000 | Max: 6843457 | Mean: 5924434.49

[Column: CREDIT_ACTIVE]
  - Unique values: 4
  - Value counts:
CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

[Column: CREDIT_CURRENCY]
  - Unique values: 4
  - Value counts:
CREDIT_CURRENCY
currency 1    1715020
currency 2       1224
currency 3        174
currency 4         10
Name: count, dtype: int64

[Column: DAYS_CREDIT]
  - Unique values: 2923
  - Min: -2922 | Max: 0 | Mean: -1142.11

[Column: CREDIT_DAY_OVERDUE]
  - Unique values: 942
  - Min: 0 | Max: 2792 | Mean: 0.82

[Column: DAYS_CREDIT_ENDDATE]
  - Unique values: 14096
  - Min: -42060.0 | Max: 31199.0 | Mean: 510.52

[Column: DAYS_ENDDATE_FACT]
  - Unique values: 2917
  - Min: -42023.0 | Max: 0.0 | Mean: -1017.44

[C

HEAD情報(bureau及びbureau_balance)より、同一キー番号のレコードが複数あることから、マージ時のデータ膨張を抑えるため集約を実施。

bureau_balanceのキー別集約  
*   STATUS  
    全件数、C件数、C率、X件数、X率  
    数値最大、数値平均、数値偏差、   
    直近のステータス
*   MONTHS_BALANCE  
    最小値、最大値、全件数



In [ ]:
#2-2　各テーブル参照、マージ検討  (bureau_balance　集約)

# 1. 準備：STATUSを数値評価用に変換（C, XはNaN、0〜5を数値化）
bb = tables['bureau_balance'].copy()
bb['STATUS_STR'] = bb['STATUS'].astype(str)
bb['STATUS_NUM'] = bb['STATUS_STR'].replace({'C': np.nan, 'X': np.nan}).astype(float)

# 2. 直近（MONTHS_BALANCEが最大）のステータスを取得
bb_latest = bb.sort_values(['SK_ID_BUREAU', 'MONTHS_BALANCE']).groupby('SK_ID_BUREAU').tail(1)
bb_latest = bb_latest[['SK_ID_BUREAU', 'STATUS_STR']].rename(columns={'STATUS_STR': 'BB_LATEST_STATUS'})

# 3. 集計実行
bb_agg = bb.groupby('SK_ID_BUREAU').agg(
    BB_STATUS_COUNT=('STATUS_STR', 'count'),
    BB_STATUS_C_COUNT=('STATUS_STR', lambda x: (x == 'C').sum()),
    BB_STATUS_X_COUNT=('STATUS_STR', lambda x: (x == 'X').sum()),
    BB_STATUS_NUM_MAX=('STATUS_NUM', 'max'),
    BB_STATUS_NUM_MEAN=('STATUS_NUM', 'mean'),
    BB_STATUS_NUM_STD=('STATUS_NUM', 'std'),
    BB_MONTHS_BALANCE_MIN=('MONTHS_BALANCE', 'min'),
    BB_MONTHS_BALANCE_MAX=('MONTHS_BALANCE', 'max'),
    BB_MONTHS_BALANCE_LEN=('MONTHS_BALANCE', 'count')
)

# 4. 率の計算 と 直近ステータスのマージ
bb_agg['BB_STATUS_C_RATIO'] = bb_agg['BB_STATUS_C_COUNT'] / bb_agg['BB_STATUS_COUNT']
bb_agg['BB_STATUS_X_RATIO'] = bb_agg['BB_STATUS_X_COUNT'] / bb_agg['BB_STATUS_COUNT']
bb_agg = bb_agg.reset_index().merge(bb_latest, on='SK_ID_BUREAU', how='left')

print(f"集約後の形状: {bb_agg.shape}")
print(bb_agg.head())

集約後の形状: (817395, 13)
   SK_ID_BUREAU  BB_STATUS_COUNT  BB_STATUS_C_COUNT  BB_STATUS_X_COUNT  \
0       5001709               97                 86                 11   
1       5001710               83                 48                 30   
2       5001711                4                  0                  1   
3       5001712               19                  9                  0   
4       5001713               22                  0                 22   

   BB_STATUS_NUM_MAX  BB_STATUS_NUM_MEAN  BB_STATUS_NUM_STD  \
0                NaN                 NaN                NaN   
1                0.0                 0.0                0.0   
2                0.0                 0.0                0.0   
3                0.0                 0.0                0.0   
4                NaN                 NaN                NaN   

   BB_MONTHS_BALANCE_MIN  BB_MONTHS_BALANCE_MAX  BB_MONTHS_BALANCE_LEN  \
0                    -96                      0                     97   
1      

headだと0とNaNのみしか見れないため、一部確認を実施

In [ ]:
#2-3　各テーブル参照、マージ検討  (bureau_balance　最大値確認)
bb_agg['BB_STATUS_NUM_MAX'].describe()

,BB_STATUS_NUM_MAX
count,687027.000000
mean,0.202825
std,0.610940
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,5.000000


MAX値で加工できていることを確認

In [ ]:
#2-4　各テーブル参照、マージ検討  結合キー確認
'''
application_train/test
previous_application(SK_ID_CURR,SK_ID_PREV)
POS_CASH_balance,instalments_payments,credit_card_balance(SK_ID_PREV)
'''
target_info = [
    ('application_train', 'SK_ID_CURR'),
    ('application_test', 'SK_ID_CURR'),
    ('previous_application', 'SK_ID_CURR'),
    ('previous_application', 'SK_ID_PREV'),
    ('POS_CASH_balance', 'SK_ID_PREV'),
    ('instalments_payments', 'SK_ID_PREV'),
    ('credit_card_balance', 'SK_ID_PREV'),
    ('bureau', 'SK_ID_CURR'),
    ('bureau', 'SK_ID_BUREAU'),
    ('bureau_balance','SK_ID_BUREAU')
]

print(f"{'Table':<25} | {'Key':<12} | {'Total Rows':>10} | {'Unique Keys':>10} | {'Repeat Rate'}")
print("-" * 80)

for table_name, key in target_info:
    df = tables[table_name]
    total_rows = len(df)
    unique_keys = df[key].nunique()
    repeat_rate = total_rows / unique_keys

    print(f"{table_name:<25} | {key:<12} | {total_rows:>10} | {unique_keys:>10} | {repeat_rate:.2f}")

Table                     | Key          | Total Rows | Unique Keys | Repeat Rate
--------------------------------------------------------------------------------
application_train         | SK_ID_CURR   |     307511 |     307511 | 1.00
application_test          | SK_ID_CURR   |      48744 |      48744 | 1.00
previous_application      | SK_ID_CURR   |    1670214 |     338857 | 4.93
previous_application      | SK_ID_PREV   |    1670214 |    1670214 | 1.00
POS_CASH_balance          | SK_ID_PREV   |   10001358 |     936325 | 10.68
instalments_payments      | SK_ID_PREV   |   13605401 |     997752 | 13.64
credit_card_balance       | SK_ID_PREV   |    3840312 |     104307 | 36.82
bureau                    | SK_ID_CURR   |    1716428 |     305811 | 5.61
bureau                    | SK_ID_BUREAU |    1716428 |    1716428 | 1.00
bureau_balance            | SK_ID_BUREAU |   27299925 |     817395 | 33.40


【キー情報整理】  
*   メインテーブルのキー重複なし  
*   previous_application,bureau  
下流テーブルとのキーでは重複ないものの、メイン結合前に集約を要する  
*   POS_CASH_balance,instalments_payments,credit_card_balance  
キー重複しているため集約を要する
  
  下流から順に集約を実施  
  ※階層順に集約→マージ→集約  
  　中間テーブルを下流テーブルとのマージ前に集約すると下流とのIDごと集約されてマージできなくなるため。


【サブテーブル(bureau_balance除く)のキー別集約　方向性】  
  
メインテーブル除くカラム数が100程度あることから  
集約時点でのカラムの深度ある検討は困難と判断。  
以下の集約処理を関数化し、一律実行を行う。
1.   カテゴリ型  
     ユニーク数、直近値
2.   数値型  
     最小、最大、平均、合計、標準偏差 、直近値  


なお、モデル実装後に更なる精度向上を図る場合においては寄与度やSHAPで重要度の高いもの、目検におけるカラム特性において有効度の高そうなものについて再考を実施。  

【修正事項】
*   カテゴリ型の最頻値については処理時間上、一旦除外して行う。
*   previous_applicationと下流テーブルをマージして集約するとセッションクラッシュするため、それぞれメインキーで集約して直接メインテーブルとマージを行う。

In [ ]:
#2-5　各テーブル参照、マージ検討  集約関数

def aggregate_table(df, group_key, sort_col, prefix=''):
    exclude = [group_key, sort_col]
    num_cols = [c for c in df.select_dtypes('number').columns if c not in exclude]
    cat_cols = [c for c in df.select_dtypes(['object', 'category']).columns if c not in exclude]

    # --- 数値型集約 ---
    num_agg = df.groupby(group_key)[num_cols].agg(['min', 'max', 'mean', 'sum', 'std'])
    num_agg.columns = [f'{prefix}{c}_{s}' for c, s in num_agg.columns]

    # --- カテゴリ型集約 ---
    if cat_cols:
        cat_agg = df.groupby(group_key)[cat_cols].agg(['nunique'])
        cat_agg.columns = [f'{prefix}{c}_{s}' for c, s in cat_agg.columns]
    else:
        cat_agg = pd.DataFrame(index=df[group_key].unique())

    # --- 直近値（sort_colが最大の行） ---
    latest = df.sort_values(sort_col).groupby(group_key).tail(1).set_index(group_key)
    latest_cols = {c: f'{prefix}{c}_latest' for c in num_cols + cat_cols}
    latest = latest[num_cols + cat_cols].rename(columns=latest_cols)

    # --- ソート列自体の集約 ---
    sort_agg = df.groupby(group_key)[sort_col].agg(['min', 'max', 'count'])
    sort_agg.columns = [f'{prefix}{sort_col}_{s}' for s in ['min', 'max', 'count']]

    # --- 結合 ---
    result = num_agg.join(cat_agg).join(latest).join(sort_agg).reset_index()
    return result


In [ ]:
#2-6　各テーブル参照、マージ検討  第3階層　集約(POS_CASH_balance)
import gc
pos_agg = aggregate_table(POS_CASH_balance, 'SK_ID_CURR', 'MONTHS_BALANCE', prefix='POS_')
del POS_CASH_balance
gc.collect()
pos_agg.head()

,SK_ID_CURR,POS_SK_ID_PREV_min,POS_SK_ID_PREV_max,POS_SK_ID_PREV_mean,POS_SK_ID_PREV_sum,POS_SK_ID_PREV_std,POS_CNT_INSTALMENT_min,POS_CNT_INSTALMENT_max,POS_CNT_INSTALMENT_mean,POS_CNT_INSTALMENT_sum,POS_CNT_INSTALMENT_std,POS_CNT_INSTALMENT_FUTURE_min,POS_CNT_INSTALMENT_FUTURE_max,POS_CNT_INSTALMENT_FUTURE_mean,POS_CNT_INSTALMENT_FUTURE_sum,POS_CNT_INSTALMENT_FUTURE_std,POS_SK_DPD_min,POS_SK_DPD_max,POS_SK_DPD_mean,POS_SK_DPD_sum,POS_SK_DPD_std,POS_SK_DPD_DEF_min,POS_SK_DPD_DEF_max,POS_SK_DPD_DEF_mean,POS_SK_DPD_DEF_sum,POS_SK_DPD_DEF_std,POS_NAME_CONTRACT_STATUS_nunique,POS_SK_ID_PREV_latest,POS_CNT_INSTALMENT_latest,POS_CNT_INSTALMENT_FUTURE_latest,POS_SK_DPD_latest,POS_SK_DPD_DEF_latest,POS_NAME_CONTRACT_STATUS_latest,POS_MONTHS_BALANCE_min,POS_MONTHS_BALANCE_max,POS_MONTHS_BALANCE_count
0,100001,1369693,1851984,1.584045e+06,14256401,254189.675833,4.0,4.0,4.000000,36.0,0.000000,0.0,4.0,1.444444,13.0,1.424001,0,7,0.777778,7,2.333333,0,7,0.777778,7,2.333333,2,1369693,4.0,0.0,0,0,Completed,-96,-53,9
1,100002,1038818,1038818,1.038818e+06,19737542,0.000000,24.0,24.0,24.000000,456.0,0.000000,6.0,24.0,15.000000,285.0,5.627315,0,0,0.000000,0,0.000000,0,0,0.000000,0,0.000000,1,1038818,24.0,6.0,0,0,Active,-19,-1,19
2,100003,1810518,2636178,2.297665e+06,64334628,329593.011850,6.0,12.0,10.107142,283.0,2.806597,0.0,12.0,5.785714,162.0,3.842811,0,0,0.000000,0,0.000000,0,0,0.000000,0,0.000000,2,1810518,7.0,0.0,0,0,Completed,-77,-18,28
3,100004,1564014,1564014,1.564014e+06,6256056,0.000000,3.0,4.0,3.750000,15.0,0.500000,0.0,4.0,2.250000,9.0,1.707825,0,0,0.000000,0,0.000000,0,0,0.000000,0,0.000000,2,1564014,3.0,0.0,0,0,Completed,-27,-24,4
4,100005,2495675,2495675,2.495675e+06,27452425,0.000000,9.0,12.0,11.700000,117.0,0.948683,0.0,12.0,7.200000,72.0,3.614784,0,0,0.000000,0,0.000000,0,0,0.000000,0,0.000000,3,2495675,9.0,0.0,0,0,Completed,-25,-15,11


In [ ]:
#2-7　各テーブル参照、マージ検討  第3階層　集約(instalments_payments)
import gc
ins_agg = aggregate_table(instalments_payments, 'SK_ID_CURR', 'DAYS_INSTALMENT', prefix='INS_')
del instalments_payments
gc.collect()
ins_agg.head()


,SK_ID_CURR,INS_SK_ID_PREV_min,INS_SK_ID_PREV_max,INS_SK_ID_PREV_mean,INS_SK_ID_PREV_sum,INS_SK_ID_PREV_std,INS_NUM_INSTALMENT_VERSION_min,INS_NUM_INSTALMENT_VERSION_max,INS_NUM_INSTALMENT_VERSION_mean,INS_NUM_INSTALMENT_VERSION_sum,INS_NUM_INSTALMENT_VERSION_std,INS_NUM_INSTALMENT_NUMBER_min,INS_NUM_INSTALMENT_NUMBER_max,INS_NUM_INSTALMENT_NUMBER_mean,INS_NUM_INSTALMENT_NUMBER_sum,INS_NUM_INSTALMENT_NUMBER_std,INS_DAYS_ENTRY_PAYMENT_min,INS_DAYS_ENTRY_PAYMENT_max,INS_DAYS_ENTRY_PAYMENT_mean,INS_DAYS_ENTRY_PAYMENT_sum,INS_DAYS_ENTRY_PAYMENT_std,INS_AMT_INSTALMENT_min,INS_AMT_INSTALMENT_max,INS_AMT_INSTALMENT_mean,INS_AMT_INSTALMENT_sum,INS_AMT_INSTALMENT_std,INS_AMT_PAYMENT_min,INS_AMT_PAYMENT_max,INS_AMT_PAYMENT_mean,INS_AMT_PAYMENT_sum,INS_AMT_PAYMENT_std,INS_SK_ID_PREV_latest,INS_NUM_INSTALMENT_VERSION_latest,INS_NUM_INSTALMENT_NUMBER_latest,INS_DAYS_ENTRY_PAYMENT_latest,INS_AMT_INSTALMENT_latest,INS_AMT_PAYMENT_latest,INS_DAYS_INSTALMENT_min,INS_DAYS_INSTALMENT_max,INS_DAYS_INSTALMENT_count
0,100001,1369693,1851984,1.576389e+06,11034724,257795.383246,1.0,2.0,1.142857,8.0,0.377964,1,4,2.714286,19,1.112697,-2916.0,-1628.0,-2195.000000,-15365.0,643.904236,3951.000000,17397.900391,5885.132324,4.119593e+04,5076.676763,3951.000000,17397.900391,5885.132324,4.119593e+04,5076.676763,1369693,2.0,4,-1628.0,17397.900391,17397.900391,-2916.0,-1619.0,7
1,100002,1038818,1038818,1.038818e+06,19737542,0.000000,1.0,2.0,1.052632,20.0,0.229416,1,19,10.000000,190,5.627314,-587.0,-49.0,-315.421051,-5993.0,172.058884,9251.775391,53093.746094,11559.247070,2.196257e+05,10058.037883,9251.775391,53093.746094,11559.247070,2.196257e+05,10058.037883,1038818,2.0,19,-49.0,53093.746094,53093.746094,-565.0,-25.0,19
2,100003,1810518,2636178,2.290070e+06,57251754,320488.923470,1.0,2.0,1.040000,26.0,0.200000,1,12,5.080000,127,3.134751,-2324.0,-544.0,-1385.319946,-34633.0,757.325439,6662.970215,560835.375000,64754.585938,1.618865e+06,110542.594872,6662.970215,560835.375000,64754.585938,1.618865e+06,110542.594872,1810518,2.0,7,-544.0,560835.375000,560835.375000,-2310.0,-536.0,25
3,100004,1564014,1564014,1.564014e+06,4692042,0.000000,1.0,2.0,1.333333,4.0,0.577350,1,3,2.000000,6,1.000000,-795.0,-727.0,-761.666687,-2285.0,34.019604,5357.250000,10573.964844,7096.154785,2.128846e+04,3011.871719,5357.250000,10573.964844,7096.154785,2.128846e+04,3011.871719,1564014,2.0,3,-727.0,10573.964844,10573.964844,-784.0,-724.0,3
4,100005,2495675,2495675,2.495675e+06,22461075,0.000000,1.0,2.0,1.111111,10.0,0.333333,1,9,5.000000,45,2.738613,-736.0,-470.0,-609.555542,-5486.0,90.554008,4813.200195,17656.244141,6240.205078,5.616184e+04,4281.014648,4813.200195,17656.244141,6240.205078,5.616184e+04,4281.014648,2495675,2.0,9,-470.0,17656.244141,17656.244141,-706.0,-466.0,9


In [ ]:
#2-8　各テーブル参照、マージ検討　第3階層　集約(credit_card_balance)
import gc
cc_agg = aggregate_table(credit_card_balance, 'SK_ID_CURR', 'MONTHS_BALANCE', prefix='CC_')
del credit_card_balance
gc.collect()
cc_agg.head()

,SK_ID_CURR,CC_SK_ID_PREV_min,CC_SK_ID_PREV_max,CC_SK_ID_PREV_mean,CC_SK_ID_PREV_sum,CC_SK_ID_PREV_std,CC_AMT_BALANCE_min,CC_AMT_BALANCE_max,CC_AMT_BALANCE_mean,CC_AMT_BALANCE_sum,CC_AMT_BALANCE_std,CC_AMT_CREDIT_LIMIT_ACTUAL_min,CC_AMT_CREDIT_LIMIT_ACTUAL_max,CC_AMT_CREDIT_LIMIT_ACTUAL_mean,CC_AMT_CREDIT_LIMIT_ACTUAL_sum,CC_AMT_CREDIT_LIMIT_ACTUAL_std,CC_AMT_DRAWINGS_ATM_CURRENT_min,CC_AMT_DRAWINGS_ATM_CURRENT_max,CC_AMT_DRAWINGS_ATM_CURRENT_mean,CC_AMT_DRAWINGS_ATM_CURRENT_sum,CC_AMT_DRAWINGS_ATM_CURRENT_std,CC_AMT_DRAWINGS_CURRENT_min,CC_AMT_DRAWINGS_CURRENT_max,CC_AMT_DRAWINGS_CURRENT_mean,CC_AMT_DRAWINGS_CURRENT_sum,CC_AMT_DRAWINGS_CURRENT_std,CC_AMT_DRAWINGS_OTHER_CURRENT_min,CC_AMT_DRAWINGS_OTHER_CURRENT_max,CC_AMT_DRAWINGS_OTHER_CURRENT_mean,CC_AMT_DRAWINGS_OTHER_CURRENT_sum,CC_AMT_DRAWINGS_OTHER_CURRENT_std,CC_AMT_DRAWINGS_POS_CURRENT_min,CC_AMT_DRAWINGS_POS_CURRENT_max,CC_AMT_DRAWINGS_POS_CURRENT_mean,CC_AMT_DRAWINGS_POS_CURRENT_sum,CC_AMT_DRAWINGS_POS_CURRENT_std,CC_AMT_INST_MIN_REGULARITY_min,CC_AMT_INST_MIN_REGULARITY_max,CC_AMT_INST_MIN_REGULARITY_mean,CC_AMT_INST_MIN_REGULARITY_sum,CC_AMT_INST_MIN_REGULARITY_std,CC_AMT_PAYMENT_CURRENT_min,CC_AMT_PAYMENT_CURRENT_max,CC_AMT_PAYMENT_CURRENT_mean,CC_AMT_PAYMENT_CURRENT_sum,CC_AMT_PAYMENT_CURRENT_std,CC_AMT_PAYMENT_TOTAL_CURRENT_min,CC_AMT_PAYMENT_TOTAL_CURRENT_max,CC_AMT_PAYMENT_TOTAL_CURRENT_mean,CC_AMT_PAYMENT_TOTAL_CURRENT_sum,...,CC_CNT_DRAWINGS_OTHER_CURRENT_min,CC_CNT_DRAWINGS_OTHER_CURRENT_max,CC_CNT_DRAWINGS_OTHER_CURRENT_mean,CC_CNT_DRAWINGS_OTHER_CURRENT_sum,CC_CNT_DRAWINGS_OTHER_CURRENT_std,CC_CNT_DRAWINGS_POS_CURRENT_min,CC_CNT_DRAWINGS_POS_CURRENT_max,CC_CNT_DRAWINGS_POS_CURRENT_mean,CC_CNT_DRAWINGS_POS_CURRENT_sum,CC_CNT_DRAWINGS_POS_CURRENT_std,CC_CNT_INSTALMENT_MATURE_CUM_min,CC_CNT_INSTALMENT_MATURE_CUM_max,CC_CNT_INSTALMENT_MATURE_CUM_mean,CC_CNT_INSTALMENT_MATURE_CUM_sum,CC_CNT_INSTALMENT_MATURE_CUM_std,CC_SK_DPD_min,CC_SK_DPD_max,CC_SK_DPD_mean,CC_SK_DPD_sum,CC_SK_DPD_std,CC_SK_DPD_DEF_min,CC_SK_DPD_DEF_max,CC_SK_DPD_DEF_mean,CC_SK_DPD_DEF_sum,CC_SK_DPD_DEF_std,CC_NAME_CONTRACT_STATUS_nunique,CC_SK_ID_PREV_latest,CC_AMT_BALANCE_latest,CC_AMT_CREDIT_LIMIT_ACTUAL_latest,CC_AMT_DRAWINGS_ATM_CURRENT_latest,CC_AMT_DRAWINGS_CURRENT_latest,CC_AMT_DRAWINGS_OTHER_CURRENT_latest,CC_AMT_DRAWINGS_POS_CURRENT_latest,CC_AMT_INST_MIN_REGULARITY_latest,CC_AMT_PAYMENT_CURRENT_latest,CC_AMT_PAYMENT_TOTAL_CURRENT_latest,CC_AMT_RECEIVABLE_PRINCIPAL_latest,CC_AMT_RECIVABLE_latest,CC_AMT_TOTAL_RECEIVABLE_latest,CC_CNT_DRAWINGS_ATM_CURRENT_latest,CC_CNT_DRAWINGS_CURRENT_latest,CC_CNT_DRAWINGS_OTHER_CURRENT_latest,CC_CNT_DRAWINGS_POS_CURRENT_latest,CC_CNT_INSTALMENT_MATURE_CUM_latest,CC_SK_DPD_latest,CC_SK_DPD_DEF_latest,CC_NAME_CONTRACT_STATUS_latest,CC_MONTHS_BALANCE_min,CC_MONTHS_BALANCE_max,CC_MONTHS_BALANCE_count
0,100006,1489396,1489396,1489396.0,8936376,0.0,0.0,0.00000,0.000000,0.00,0.000000,270000,270000,270000.000000,1620000,0.000000,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.000000,0.0,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,0.00,NaN,0.0,0.0,0.000000,0.0000,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.000000,0.0,0.000000,0,0,0.000000,0,0.000000,0,0,0.000000,0,0.000000,1,1489396,0.0,270000,NaN,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,0.0,0,0,Active,-6,-1,6
1,100011,1843384,1843384,1843384.0,136410416,0.0,0.0,189000.00000,54482.113281,4031676.25,68127.238133,90000,180000,164189.189189,12150000,34482.743620,0.0,180000.0,2432.432373,180000.0,20924.574974,0.0,180000.0,2432.432373,180000.0,20924.574974,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9000.0,3956.221680,288804.187500,4487.750710,0.0,55485.0,4843.063965,358386.75,7279.601961,0.0,55485.0,4520.067383,334485.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,33.0,25.767124,1881.0,10.288236,0,0,0.000000,0,0.000000,0,0,0.000000,0,0.000000,1,1843384,0.0,90000,0.0,0.0,0.0,0.0,0.0,563.354980,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,33.0,0,0,Active,-75,-2,74
2,100

In [ ]:
#2-9　各テーブル参照、マージ検討 結合キー再確認
'''
application_train/test
previous_application(SK_ID_CURR,SK_ID_PREV)
POS_CASH_balance,instalments_payments,credit_card_balance(SK_ID_PREV)
'''
# テーブル名、データフレーム変数、チェックしたいキーをセットにする
target_agg = [
    ('application_train',    application_train,    'SK_ID_CURR'),
    ('application_test',     application_test,     'SK_ID_CURR'),
    ('previous_application', previous_application, 'SK_ID_CURR'),
    ('pos_agg',              pos_agg,              'SK_ID_CURR'),
    ('ins_agg',              ins_agg,              'SK_ID_CURR'),
    ('cc_agg',               cc_agg,               'SK_ID_CURR'),
    ('bureau',               bureau,               'SK_ID_CURR'),
    ('bureau',               bureau,               'SK_ID_BUREAU'),
    ('bb_agg',               bb_agg,               'SK_ID_BUREAU')
]

print(f"{'Table':<25} | {'Key':<12} | {'Total Rows':>10} | {'Unique Keys':>10} | {'Repeat Rate'}")
print("-" * 80)

for name, df, key in target_agg:
    total_rows = len(df)
    unique_keys = df[key].nunique()
    repeat_rate = total_rows / unique_keys

    print(f"{name:<25} | {key:<12} | {total_rows:>10} | {unique_keys:>10} | {repeat_rate:.2f}")

Table                     | Key          | Total Rows | Unique Keys | Repeat Rate
--------------------------------------------------------------------------------
application_train         | SK_ID_CURR   |     307511 |     307511 | 1.00
application_test          | SK_ID_CURR   |      48744 |      48744 | 1.00
previous_application      | SK_ID_CURR   |    1670214 |     338857 | 4.93
pos_agg                   | SK_ID_CURR   |     337252 |     337252 | 1.00
ins_agg                   | SK_ID_CURR   |     339587 |     339587 | 1.00
cc_agg                    | SK_ID_CURR   |     103558 |     103558 | 1.00
bureau                    | SK_ID_CURR   |    1716428 |     305811 | 5.61
bureau                    | SK_ID_BUREAU |    1716428 |    1716428 | 1.00
bb_agg                    | SK_ID_BUREAU |     817395 |     817395 | 1.00


第2階層と第3階層のキー(SK_ID_BUREAU)が1対1であることを確認

In [ ]:
#3-1　マージ 第2-3階層　bureau-bureaubalance　(SK_ID_BUREAU)

bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
del bb_agg
gc.collect()

print(f"bureau shape: {bureau.shape}")

bureau shape: (1716428, 29)


In [ ]:
'''以下のマージ後のテーブルでは、メインキー集約時にセッションクラッシュのため使用不可'''

#　マージ　第2-3階層　previous_application-POS_CASH_balance,
#                                         -instalments_payments,
#                                         -credit_card_balance　(SK_ID_PREV)

# previous_application = previous_application.merge(pos_agg, on='SK_ID_PREV', how='left')
# del pos_agg
# previous_application = previous_application.merge(ins_agg, on='SK_ID_PREV', how='left')
# del ins_agg
# previous_application = previous_application.merge(cc_agg,  on='SK_ID_PREV', how='left')
# del cc_agg
# gc.collect()

# print(f"New previous_application shape: {previous_application.shape}")

'以下のマージ後のテーブルでは、メインキー集約時にセッションクラッシュのため使用不可'

In [ ]:
#3-2　マージ　第2階層　集約(bureau)
import gc
bureau_agg = aggregate_table(bureau, 'SK_ID_CURR', 'DAYS_CREDIT', prefix='BU_')
del bureau
gc.collect()
bureau_agg.head()

,SK_ID_CURR,BU_SK_ID_BUREAU_min,BU_SK_ID_BUREAU_max,BU_SK_ID_BUREAU_mean,BU_SK_ID_BUREAU_sum,BU_SK_ID_BUREAU_std,BU_CREDIT_DAY_OVERDUE_min,BU_CREDIT_DAY_OVERDUE_max,BU_CREDIT_DAY_OVERDUE_mean,BU_CREDIT_DAY_OVERDUE_sum,BU_CREDIT_DAY_OVERDUE_std,BU_DAYS_CREDIT_ENDDATE_min,BU_DAYS_CREDIT_ENDDATE_max,BU_DAYS_CREDIT_ENDDATE_mean,BU_DAYS_CREDIT_ENDDATE_sum,BU_DAYS_CREDIT_ENDDATE_std,BU_DAYS_ENDDATE_FACT_min,BU_DAYS_ENDDATE_FACT_max,BU_DAYS_ENDDATE_FACT_mean,BU_DAYS_ENDDATE_FACT_sum,BU_DAYS_ENDDATE_FACT_std,BU_AMT_CREDIT_MAX_OVERDUE_min,BU_AMT_CREDIT_MAX_OVERDUE_max,BU_AMT_CREDIT_MAX_OVERDUE_mean,BU_AMT_CREDIT_MAX_OVERDUE_sum,BU_AMT_CREDIT_MAX_OVERDUE_std,BU_CNT_CREDIT_PROLONG_min,BU_CNT_CREDIT_PROLONG_max,BU_CNT_CREDIT_PROLONG_mean,BU_CNT_CREDIT_PROLONG_sum,BU_CNT_CREDIT_PROLONG_std,BU_AMT_CREDIT_SUM_min,BU_AMT_CREDIT_SUM_max,BU_AMT_CREDIT_SUM_mean,BU_AMT_CREDIT_SUM_sum,BU_AMT_CREDIT_SUM_std,BU_AMT_CREDIT_SUM_DEBT_min,BU_AMT_CREDIT_SUM_DEBT_max,BU_AMT_CREDIT_SUM_DEBT_mean,BU_AMT_CREDIT_SUM_DEBT_sum,BU_AMT_CREDIT_SUM_DEBT_std,BU_AMT_CREDIT_SUM_LIMIT_min,BU_AMT_CREDIT_SUM_LIMIT_max,BU_AMT_CREDIT_SUM_LIMIT_mean,BU_AMT_CREDIT_SUM_LIMIT_sum,BU_AMT_CREDIT_SUM_LIMIT_std,BU_AMT_CREDIT_SUM_OVERDUE_min,BU_AMT_CREDIT_SUM_OVERDUE_max,BU_AMT_CREDIT_SUM_OVERDUE_mean,BU_AMT_CREDIT_SUM_OVERDUE_sum,...,BU_BB_MONTHS_BALANCE_MAX_std,BU_BB_MONTHS_BALANCE_LEN_min,BU_BB_MONTHS_BALANCE_LEN_max,BU_BB_MONTHS_BALANCE_LEN_mean,BU_BB_MONTHS_BALANCE_LEN_sum,BU_BB_MONTHS_BALANCE_LEN_std,BU_BB_STATUS_C_RATIO_min,BU_BB_STATUS_C_RATIO_max,BU_BB_STATUS_C_RATIO_mean,BU_BB_STATUS_C_RATIO_sum,BU_BB_STATUS_C_RATIO_std,BU_BB_STATUS_X_RATIO_min,BU_BB_STATUS_X_RATIO_max,BU_BB_STATUS_X_RATIO_mean,BU_BB_STATUS_X_RATIO_sum,BU_BB_STATUS_X_RATIO_std,BU_CREDIT_ACTIVE_nunique,BU_CREDIT_CURRENCY_nunique,BU_CREDIT_TYPE_nunique,BU_BB_LATEST_STATUS_nunique,BU_SK_ID_BUREAU_latest,BU_CREDIT_DAY_OVERDUE_latest,BU_DAYS_CREDIT_ENDDATE_latest,BU_DAYS_ENDDATE_FACT_latest,BU_AMT_CREDIT_MAX_OVERDUE_latest,BU_CNT_CREDIT_PROLONG_latest,BU_AMT_CREDIT_SUM_latest,BU_AMT_CREDIT_SUM_DEBT_latest,BU_AMT_CREDIT_SUM_LIMIT_latest,BU_AMT_CREDIT_SUM_OVERDUE_latest,BU_DAYS_CREDIT_UPDATE_latest,BU_AMT_ANNUITY_latest,BU_BB_STATUS_COUNT_latest,BU_BB_STATUS_C_COUNT_latest,BU_BB_STATUS_X_COUNT_latest,BU_BB_STATUS_NUM_MAX_latest,BU_BB_STATUS_NUM_MEAN_latest,BU_BB_STATUS_NUM_STD_latest,BU_BB_MONTHS_BALANCE_MIN_latest,BU_BB_MONTHS_BALANCE_MAX_latest,BU_BB_MONTHS_BALANCE_LEN_latest,BU_BB_STATUS_C_RATIO_latest,BU_BB_STATUS_X_RATIO_latest,BU_CREDIT_ACTIVE_latest,BU_CREDIT_CURRENCY_latest,BU_CREDIT_TYPE_latest,BU_BB_LATEST_STATUS_latest,BU_DAYS_CREDIT_min,BU_DAYS_CREDIT_max,BU_DAYS_CREDIT_count
0,100001,5896630,5896636,5896633.000,41276431,2.160247,0,0,0.0,0,0.0,-1329.0,1778.0,82.428574,577.0,1032.859277,-1328.0,-544.0,-825.500000,-3302.0,369.078583,NaN,NaN,NaN,0.000000,NaN,0,0,0.0,0,0.0,85500.0,378000.000000,207623.578125,1.453365e+06,122544.544510,0.0,373239.0,85240.929688,596686.5,137485.631124,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.000000,2.0,52.0,24.571429,172.0,16.050515,0.0,0.966667,0.441240,3.088683,0.428578,0.0,0.500000,0.214590,1.502129,0.182611,2,1,1,3,5896635,0,1778.0,NaN,NaN,0,378000.000000,373239.0,0.000000,0.0,-16,10822.5,2.0,0.0,1.0,0.0,0.0,NaN,-1.0,0.0,2.0,0.0,0.500000,Active,currency 1,Consumer credit,0,-1572,-49,7
1,100002,6113835,6158909,6153272.125,49226177,15935.004993,0,0,0.0,0,0.0,-1072.0,780.0,-349.000000,-2094.0,767.490977,-1185.0,-36.0,-697.500000,-4185.0,515.992554,0.0,5043.64502,1681.028931,8405.144531,2363.246907,0,0,0.0,0,0.0,0.0,450000.000000,108131.945312,8.650556e+05,146075.557476,0.0,245781.0,49156.199219,245781.0,109916.604716,0.0,31988.564453,7997.141113,31988.564453,15994.282227,0.0,0.0,0.0,0.0,...,10.783585,4.0,22.0,13.750000,110.0,6.363961,0.0,0.812500,0.175426,1.403409,0.263147,0.0,0.500000,0.161932,1.295455,0.161650,2,1,2,2,6158909,0,NaN,NaN,40.5,0,31988.564453,0.0,31988.564453,0.0,-24,0.0,4.0,0.0,2.0,0.0,0.0,0.0,-3.0,0.0,4.0,0.0,0.500000,Active,currency 

In [ ]:
#3-3　マージ　第2階層　集約(previous_application)

import gc
prev_agg = aggregate_table(previous_application, 'SK_ID_CURR', 'DAYS_DECISION', prefix='PREV_')
del previous_application
gc.collect()
prev_agg.head()


,SK_ID_CURR,PREV_SK_ID_PREV_min,PREV_SK_ID_PREV_max,PREV_SK_ID_PREV_mean,PREV_SK_ID_PREV_sum,PREV_SK_ID_PREV_std,PREV_AMT_ANNUITY_min,PREV_AMT_ANNUITY_max,PREV_AMT_ANNUITY_mean,PREV_AMT_ANNUITY_sum,PREV_AMT_ANNUITY_std,PREV_AMT_APPLICATION_min,PREV_AMT_APPLICATION_max,PREV_AMT_APPLICATION_mean,PREV_AMT_APPLICATION_sum,PREV_AMT_APPLICATION_std,PREV_AMT_CREDIT_min,PREV_AMT_CREDIT_max,PREV_AMT_CREDIT_mean,PREV_AMT_CREDIT_sum,PREV_AMT_CREDIT_std,PREV_AMT_DOWN_PAYMENT_min,PREV_AMT_DOWN_PAYMENT_max,PREV_AMT_DOWN_PAYMENT_mean,PREV_AMT_DOWN_PAYMENT_sum,PREV_AMT_DOWN_PAYMENT_std,PREV_AMT_GOODS_PRICE_min,PREV_AMT_GOODS_PRICE_max,PREV_AMT_GOODS_PRICE_mean,PREV_AMT_GOODS_PRICE_sum,PREV_AMT_GOODS_PRICE_std,PREV_HOUR_APPR_PROCESS_START_min,PREV_HOUR_APPR_PROCESS_START_max,PREV_HOUR_APPR_PROCESS_START_mean,PREV_HOUR_APPR_PROCESS_START_sum,PREV_HOUR_APPR_PROCESS_START_std,PREV_NFLAG_LAST_APPL_IN_DAY_min,PREV_NFLAG_LAST_APPL_IN_DAY_max,PREV_NFLAG_LAST_APPL_IN_DAY_mean,PREV_NFLAG_LAST_APPL_IN_DAY_sum,PREV_NFLAG_LAST_APPL_IN_DAY_std,PREV_RATE_DOWN_PAYMENT_min,PREV_RATE_DOWN_PAYMENT_max,PREV_RATE_DOWN_PAYMENT_mean,PREV_RATE_DOWN_PAYMENT_sum,PREV_RATE_DOWN_PAYMENT_std,PREV_RATE_INTEREST_PRIMARY_min,PREV_RATE_INTEREST_PRIMARY_max,PREV_RATE_INTEREST_PRIMARY_mean,PREV_RATE_INTEREST_PRIMARY_sum,...,PREV_NAME_CONTRACT_STATUS_nunique,PREV_NAME_PAYMENT_TYPE_nunique,PREV_CODE_REJECT_REASON_nunique,PREV_NAME_TYPE_SUITE_nunique,PREV_NAME_CLIENT_TYPE_nunique,PREV_NAME_GOODS_CATEGORY_nunique,PREV_NAME_PORTFOLIO_nunique,PREV_NAME_PRODUCT_TYPE_nunique,PREV_CHANNEL_TYPE_nunique,PREV_NAME_SELLER_INDUSTRY_nunique,PREV_NAME_YIELD_GROUP_nunique,PREV_PRODUCT_COMBINATION_nunique,PREV_SK_ID_PREV_latest,PREV_AMT_ANNUITY_latest,PREV_AMT_APPLICATION_latest,PREV_AMT_CREDIT_latest,PREV_AMT_DOWN_PAYMENT_latest,PREV_AMT_GOODS_PRICE_latest,PREV_HOUR_APPR_PROCESS_START_latest,PREV_NFLAG_LAST_APPL_IN_DAY_latest,PREV_RATE_DOWN_PAYMENT_latest,PREV_RATE_INTEREST_PRIMARY_latest,PREV_RATE_INTEREST_PRIVILEGED_latest,PREV_SELLERPLACE_AREA_latest,PREV_CNT_PAYMENT_latest,PREV_DAYS_FIRST_DRAWING_latest,PREV_DAYS_FIRST_DUE_latest,PREV_DAYS_LAST_DUE_1ST_VERSION_latest,PREV_DAYS_LAST_DUE_latest,PREV_DAYS_TERMINATION_latest,PREV_NFLAG_INSURED_ON_APPROVAL_latest,PREV_NAME_CONTRACT_TYPE_latest,PREV_WEEKDAY_APPR_PROCESS_START_latest,PREV_FLAG_LAST_APPL_PER_CONTRACT_latest,PREV_NAME_CASH_LOAN_PURPOSE_latest,PREV_NAME_CONTRACT_STATUS_latest,PREV_NAME_PAYMENT_TYPE_latest,PREV_CODE_REJECT_REASON_latest,PREV_NAME_TYPE_SUITE_latest,PREV_NAME_CLIENT_TYPE_latest,PREV_NAME_GOODS_CATEGORY_latest,PREV_NAME_PORTFOLIO_latest,PREV_NAME_PRODUCT_TYPE_latest,PREV_CHANNEL_TYPE_latest,PREV_NAME_SELLER_INDUSTRY_latest,PREV_NAME_YIELD_GROUP_latest,PREV_PRODUCT_COMBINATION_latest,PREV_DAYS_DECISION_min,PREV_DAYS_DECISION_max,PREV_DAYS_DECISION_count
0,100001,1369693,1369693,1.369693e+06,1369693,NaN,3951.000000,3951.000000,3951.000000,3951.000000,NaN,24835.5,24835.5,24835.50,24835.5,NaN,23787.0,23787.0,23787.00,23787.0,NaN,2520.0,2520.0,2520.0,2520.0,NaN,24835.5,24835.5,24835.5,24835.5,NaN,13,13,13.000000,13,NaN,1,1,1.0,1,NaN,0.104326,0.104326,0.104326,0.104326,NaN,NaN,NaN,NaN,0.0,...,1,1,1,1,1,1,1,1,1,1,1,1,1369693,3951.000000,24835.5,23787.0,2520.0,24835.5,13,1,0.104326,NaN,NaN,23,8.0,365243.0,-1709.0,-1499.0,-1619.0,-1612.0,0.0,Consumer loans,FRIDAY,Y,XAP,Approved,Cash through the bank,XAP,Family,Refreshed,Mobile,POS,XNA,Country-wide,Connectivity,high,POS mobile with interest,-1740,-1740,1
1,100002,1038818,1038818,1.038818e+06,1038818,NaN,9251.775391,9251.775391,9251.775391,9251.775391,NaN,179055.0,179055.0,179055.00,179055.0,NaN,179055.0,179055.0,179055.00,179055.0,NaN,0.0,0.0,0.0,0.0,NaN,179055.0,179055.0,179055.0,179055.0,NaN,9,9,9.000000,9,NaN,1,1,1.0,1,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.0,...,1,1,1,0,1,1,1,1,1,1,1,1,1038818,9251.775391,179055.0,179055.0,0.0,179055.0,9,1,0.000000,NaN,NaN,500,24.0,365243.0,-565.0,125.0,-25.0,-17.0,0.0,Consumer loans,SATURDAY,Y,XAP,Approved,XNA,XAP,NaN,New,

In [ ]:
#3-4　マージ及びダウンキャスト　第1-2、第1-3階層
# application_train/test - bureau
#　    (SK_ID_CURR)        - previous_application
#                          -POS_CASH_balance,
#                          -instalments_payments,
#                          -credit_card_balance

application_train = application_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
application_test  = application_test.merge(bureau_agg, on='SK_ID_CURR', how='left')
del bureau_agg
application_train = application_train.merge(prev_agg, on='SK_ID_CURR', how='left')
application_test  = application_test.merge(prev_agg, on='SK_ID_CURR', how='left')
del prev_agg
application_train = application_train.merge(pos_agg, on='SK_ID_CURR', how='left')
application_test  = application_test.merge(pos_agg, on='SK_ID_CURR', how='left')
del pos_agg
application_train = application_train.merge(ins_agg, on='SK_ID_CURR', how='left')
application_test  = application_test.merge(ins_agg, on='SK_ID_CURR', how='left')
del ins_agg
application_train = application_train.merge(cc_agg, on='SK_ID_CURR', how='left')
application_test  = application_test.merge(cc_agg, on='SK_ID_CURR', how='left')
del cc_agg

gc.collect()
application_train = reduce(application_train)
application_test  = reduce(application_test)

メモリ使用量: 941.40MB -> 661.03MB |  削減: 0.30MB 29.8%
メモリ使用量: 149.19MB -> 104.75MB |  削減: 0.30MB 29.8%


In [ ]:
#4-1　AMTリサイズ(データ加工時の桁あふれ対策)

train_test_table = [application_train, application_test]
names = ["application_train", "application_test"]

for i, df in enumerate(train_test_table):
    amt_cols = df.filter(like='AMT').columns.tolist()

    if len(amt_cols) > 0:
        df[amt_cols] = df[amt_cols].astype(np.float64)

        print(f"Updated {names[i]}: {len(amt_cols)} columns converted to float64")
    else:
        continue

print("\n--- Check application_train types (AMT only) ---")
print(application_train.filter(like='AMT').dtypes.head())

# 5. メモリの最終確認
import gc
gc.collect()
print(f"\nTrain Memory: {application_train.memory_usage().sum() / 1024**2:.2f} MB")

Updated application_train: 160 columns converted to float64
Updated application_test: 160 columns converted to float64

--- Check application_train types (AMT only) ---
AMT_INCOME_TOTAL              float64
AMT_CREDIT                    float64
AMT_ANNUITY                   float64
AMT_GOODS_PRICE               float64
AMT_REQ_CREDIT_BUREAU_HOUR    float64
dtype: object

Train Memory: 848.72 MB


In [ ]:
#4-2　データ型再確認
# Train と Test の形状を確認
print(f"Train Shape: {application_train.shape}")
print(f"Test Shape:  {application_test.shape}")
# データ型の内訳を確認
print("Train Data Types Summary:")
print(application_train.dtypes.value_counts())
print(application_test.dtypes.value_counts())
# 初期行数: Train=307511, Test=48744

Train Shape: (307511, 619)
Test Shape:  (48744, 618)
Train Data Types Summary:
float32     380
float64     160
int8         37
category      3
category      2
int16         2
int32         2
category      2
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
object        1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64
float32     380
float64     160
int8         36
category      3
int32         2
int16         2
category      2
category      2
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1

集約時にオブジェクト型が混ざったため、再変換

In [ ]:
#4-3　オブジェクト変換
# 変換対象のデータフレームをリストにする
dfs = [application_train, application_test]

for df in dfs:
    # 各データフレームごとに object 型の列を抽出
    obj_cols = df.select_dtypes('object').columns
    # 抽出した列を一括で category 型に変換
    df[obj_cols] = df[obj_cols].astype('category')
    # 変換されたカラム名が数値型でないことの確認
    print(f"Converted columns: {list(obj_cols)}")

Converted columns: ['BU_BB_LATEST_STATUS_latest']
Converted columns: ['BU_BB_LATEST_STATUS_latest']


In [ ]:
#5-1　モデル試走(LightGBM/CatBoost)　関数

!pip install catboost
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# --- 2. [重要] drop_list の定義 ---
# カラム削除後の再利用のため
if 'drop_list' not in globals():
    drop_list = []

# --- 3. データの準備 ---
y = application_train['TARGET']
X = application_train.drop(columns=['TARGET', 'SK_ID_CURR'] + [c for c in drop_list if c in application_train.columns], errors='ignore')

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_features = X_train.select_dtypes(include=['category']).columns.tolist()

params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'random_state': 42,
    'learning_rate': 0.05,
    'n_estimators': 100
}

# --- 4. 関数の定義 ---
def run_lgb_trial(X_train, y_train, X_valid, y_valid, params):
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.log_evaluation(period=0), lgb.early_stopping(stopping_rounds=50)]
    )
    auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])
    imp = pd.DataFrame({
        'feature': X_train.columns,
        'LGBM_pct': 100 * model.feature_importances_ / model.feature_importances_.sum()
    })
    return model, auc, imp

def run_cat_trial(X_train, y_train, X_valid, y_valid, categorical_features):
    xt_cb = X_train.copy()
    xv_cb = X_valid.copy()
    for col in categorical_features:
        xt_cb[col] = xt_cb[col].astype(str).replace('nan', 'Unknown')
        xv_cb[col] = xv_cb[col].astype(str).replace('nan', 'Unknown')

    model = cb.CatBoostClassifier(
        iterations=300, learning_rate=0.05, depth=6, eval_metric='AUC',
        random_seed=42, logging_level='Silent',
        cat_features=categorical_features, task_type='CPU'
    )
    model.fit(xt_cb, y_train, eval_set=(xv_cb, y_valid), early_stopping_rounds=50)
    auc = roc_auc_score(y_valid, model.predict_proba(xv_cb)[:, 1])
    imp = pd.DataFrame({
        'feature': X_train.columns,
        'CAT_pct': 100 * model.get_feature_importance() / model.get_feature_importance().sum()
    })
    return model, auc, imp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.0 MB/s eta 0:00:00


In [ ]:
#5-2　モデル試走(LightGBM/CatBoost)　試走
print(" LightGBM 試走開始")
lgb_model, lgb_auc, lgb_imp = run_lgb_trial(X_train, y_train, X_valid, y_valid, params)

print(" CatBoost 試走開始")
cat_model, cat_auc, cat_imp = run_cat_trial(X_train, y_train, X_valid, y_valid, categorical_features)

print(f"\n 試走完了 AUC - LGBM: {lgb_auc:.4f} / CatBoost: {cat_auc:.4f}")

 LightGBM 試走開始
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.777094
 CatBoost 試走開始

 試走完了 AUC - LGBM: 0.7771 / CatBoost: 0.7817


In [ ]:
#5-3　モデル試走(LightGBM/CatBoost)　試走結果
# --- 3. 比較用データの作成 ---

# マージして比較表を作成
comparison_df =pd.merge(lgb_imp, cat_imp, on='feature').sort_values(by='LGBM_pct', ascending=False)

# --- 4. 結果表示 ---
print(f"　 精度比較 (AUC):")
print(f"  - LightGBM: {lgb_auc:.4f}")
print(f"  - CatBoost: {cat_auc:.4f}\n")

# LGBMのTOP15
lgb_top15 = comparison_df.sort_values(by='LGBM_pct', ascending=False).head(15)[['feature', 'LGBM_pct']]
lgb_top15.columns = ['LGBM Feature', 'Importance(%)']

# CatBoostのTOP15
cb_top15 = comparison_df.sort_values(by='CAT_pct', ascending=False).head(15)[['feature', 'CAT_pct']]
cb_top15.columns = ['CatBoost Feature', 'Importance(%)']

# インデックスをリセットして横に並べる
lgb_top15 = lgb_top15.reset_index(drop=True)
cb_top15 = cb_top15.reset_index(drop=True)
side_by_side = pd.concat([lgb_top15, cb_top15], axis=1)

print(" モデル別インポータンス TOP 15")
print(side_by_side.to_markdown(index=False))

# --- 共通の注目株 ---
common_top = set(lgb_top15['LGBM Feature']) & set(cb_top15['CatBoost Feature'])
print(f"\n 両モデルが共通してTOP15に挙げた項目: {len(common_top)}件")
print(list(common_top))


　 精度比較 (AUC):
  - LightGBM: 0.7771
  - CatBoost: 0.7817

 モデル別インポータンス TOP 15
| LGBM Feature                     |   Importance(%) | CatBoost Feature                |   Importance(%) |
|:---------------------------------|----------------:|:--------------------------------|----------------:|
| ORGANIZATION_TYPE                |         8.4     | EXT_SOURCE_3                    |        16.7896  |
| EXT_SOURCE_2                     |         5.86667 | EXT_SOURCE_2                    |        15.7277  |
| EXT_SOURCE_1                     |         5.76667 | EXT_SOURCE_1                    |         5.56949 |
| EXT_SOURCE_3                     |         5.03333 | DAYS_BIRTH                      |         3.0277  |
| AMT_ANNUITY                      |         2.7     | AMT_GOODS_PRICE                 |         2.36075 |
| INS_AMT_PAYMENT_min              |         2.26667 | CODE_GENDER                     |         2.21707 |
| BU_DAYS_CREDIT_max               |         2.2     | AMT_ANNUITY 

In [ ]:
#6-1　リークの検討
'''
割合としては特段強いシグナルを持つカラムはなく、
5％を超えるものとしては外部データ(EXT_SOURCE)と職業( ORGANIZATION_TYPE)であり、リークではないと判断。
'''
#7-1　モデル再試走
'''
リークはないと判断したため、手順省略
'''

'\nリークはないと判断したため、手順省略\n'

In [ ]:
#8-1　両モデル共通の寄与度0カラム削除(X_train,X_valid)
'''
後のEDAで使用するため中止
'''
# # --- 1. 不要列リストの確定 ---
# common_zero = comparison_df[(comparison_df['LGBM_pct'] == 0) & (comparison_df['CAT_pct'] == 0)]
# drop_list = common_zero['feature'].tolist()
# print(f" 共通寄与度0の項目数: {len(drop_list)} / {len(comparison_df)}")

# # --- 2. 全データセット（Train/Valid/Test）から一括ドロップ ---
# # X_test が定義されていることを前提としています
# X_train = X_train.drop(columns=drop_list)
# X_valid = X_valid.drop(columns=drop_list)

# # テストデータも同様に処理（もし既に存在すれば）
# if 'X_test' in globals():
#     X_test = X_test.drop(columns=drop_list)
#     print(" X_test 不要列削除")

# # カテゴリ変数リストも同期
# categorical_features = [c for c in categorical_features if c in X_train.columns]

# # --- 3. 最終結果の確認 ---
# print("-" * 30)
# print(f" 削除されたカラム数: {len(drop_list)}")
# print(f" スリム化後のカラム数 (X_train): {X_train.shape[1]}")
# if 'X_test' in globals():
#     print(f" スリム化後のカラム数 (X_test) : {X_test.shape[1]}")
# print(f" カテゴリ変数リストの残り数: {len(categorical_features)}")
# print("-" * 30)

# # 削除対象された特徴量の全リストを確認
# print(f"削除された特徴量の数: {len(drop_list)}")
# print("-" * 30)

# for i in range(0, len(drop_list), 5):
#     # 5つずつスライスして取り出す
#     chunk = drop_list[i:i+5]
#     print("".join(f"{col:<30}" for col in chunk))

 共通寄与度0の項目数: 181 / 617
------------------------------
 削除されたカラム数: 181
 スリム化後のカラム数 (X_train): 436
 カテゴリ変数リストの残り数: 22
------------------------------
削除された特徴量の数: 181
------------------------------
CC_SK_DPD_DEF_latest          FLAG_DOCUMENT_20              PREV_DAYS_DECISION_count      FLAG_DOCUMENT_21              AMT_REQ_CREDIT_BUREAU_HOUR    
AMT_REQ_CREDIT_BUREAU_DAY     AMT_REQ_CREDIT_BUREAU_WEEK    AMT_REQ_CREDIT_BUREAU_MON     CC_CNT_INSTALMENT_MATURE_CUM_latestAMT_REQ_CREDIT_BUREAU_YEAR    
CC_CNT_INSTALMENT_MATURE_CUM_minNONLIVINGAREA_MEDI            FONDKAPREMONT_MODE            PREV_NAME_PORTFOLIO_latest    NAME_HOUSING_TYPE             
BU_SK_ID_BUREAU_std           CC_AMT_DRAWINGS_OTHER_CURRENT_latestCC_NAME_CONTRACT_STATUS_nuniquePOS_SK_DPD_DEF_min            POS_NAME_CONTRACT_STATUS_nunique
REG_REGION_NOT_LIVE_REGION    REG_REGION_NOT_WORK_REGION    LIVE_REGION_NOT_WORK_REGION   REG_CITY_NOT_WORK_CITY        CC_MONTHS_BALANCE_max         
FLAG_DOCUMENT_17              FLAG_

In [ ]:
#9-1　モデル再試走(LightGBM/CATBoost)

current_cols = X_train.shape[1]
print(f"\n LightGBM ({current_cols}列) 試走開始")
lgb_model, lgb_auc, lgb_imp = run_lgb_trial(X_train, y_train, X_valid, y_valid, params)

print(f"\n CatBoost ({current_cols}列) 試走開始")
cat_model, cat_auc, cat_imp = run_cat_trial(X_train, y_train, X_valid, y_valid, categorical_features)

print("\n" + "="*30)
print(f" 再試走スコア (AUC)")
print(f"  - LightGBM : {lgb_auc:.4f}")
print(f"  - CatBoost  : {cat_auc:.4f}")
print("="*30)


 LightGBM (436列) 試走開始
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.777094

 CatBoost (436列) 試走開始

 再試走スコア (AUC)
  - LightGBM : 0.7771
  - CatBoost  : 0.7820


In [ ]:
#9-2　モデル再試走(LightGBM/CatBoost)　試走結果
# --- 3. 比較用データの作成 ---

# マージして比較表を作成
comparison_df =pd.merge(lgb_imp, cat_imp, on='feature').sort_values(by='LGBM_pct', ascending=False)

# --- 4. 結果表示 ---
print(f" 精度比較 (AUC) [スリム版]:")
print(f"  - LightGBM: {lgb_auc:.4f}")
print(f"  - CatBoost: {cat_auc:.4f}\n")

# LGBMのTOP15
lgb_top15_s = comparison_df.sort_values(by='LGBM_pct', ascending=False).head(15)[['feature', 'LGBM_pct']]
lgb_top15_s.columns = ['LGBM Feature', 'Importance(%)']

# CatBoostのTOP15
cb_top15_s = comparison_df.sort_values(by='CAT_pct', ascending=False).head(15)[['feature', 'CAT_pct']]
cb_top15_s.columns = ['CatBoost Feature', 'Importance(%)']

# インデックスをリセットして横に並べる
lgb_top15_s = lgb_top15_s.reset_index(drop=True)
cb_top15_s = cb_top15_s.reset_index(drop=True)
side_by_side_s = pd.concat([lgb_top15_s, cb_top15_s], axis=1)

print(" モデル別インポータンス TOP 15 (436列版)")
print(side_by_side_s.to_markdown(index=False))

# --- 共通 ---
common_top_s = set(lgb_top15_s['LGBM Feature']) & set(cb_top15_s['CatBoost Feature'])
print(f"\n 両モデルが共通してTOP15に挙げた項目: {len(common_top_s)}件")
print(sorted(list(common_top_s)))


 精度比較 (AUC) [スリム版]:
  - LightGBM: 0.7771
  - CatBoost: 0.7820

 モデル別インポータンス TOP 15 (436列版)
| LGBM Feature                     |   Importance(%) | CatBoost Feature            |   Importance(%) |
|:---------------------------------|----------------:|:----------------------------|----------------:|
| ORGANIZATION_TYPE                |         8.4     | EXT_SOURCE_3                |        16.7332  |
| EXT_SOURCE_2                     |         5.86667 | EXT_SOURCE_2                |        16.1962  |
| EXT_SOURCE_1                     |         5.76667 | EXT_SOURCE_1                |         5.1853  |
| EXT_SOURCE_3                     |         5.03333 | DAYS_BIRTH                  |         3.30397 |
| AMT_ANNUITY                      |         2.7     | AMT_GOODS_PRICE             |         2.3219  |
| INS_AMT_PAYMENT_min              |         2.26667 | AMT_ANNUITY                 |         2.31484 |
| BU_DAYS_CREDIT_max               |         2.2     | CODE_GENDER                 | 

寄与度整理
*   第1有力指標 (5%以上)  
共通： EXT_SOURCE_1～3 (外部機関によるスコア)  
LGBM： ORGANIZATION_TYPE (勤務先の業種)  

*   第2有力指標 (2～5%)  
共通： AMT_ANNUITY (ローンの年間支払い額)  
LGBM： INS_AMT_PAYMENT_min, BU_DAYS_CREDIT_max, AMT_CREDIT  
CAT： DAYS_BIRTH (年齢), AMT_GOODS_PRICE, CODE_GENDER  

*   その他 (重要度TOP 15圏内、上記との重複除く)  
共通：INS_AMT_PAYMENT_sum (分割払いの総支払額)  
LGBM：  
　OWN_CAR_AGE (所有車の年数)  
　INS_DAYS_ENTRY_PAYMENT_std (支払い日のバラつき/遅延傾向)  
　POS_SK_DPD_DEF_mean (POSローンの延滞日数)  
　CC_CNT_DRAWINGS_ATM_CURRENT_mean (クレカのATM引き出し回数)  
CAT：  
　NAME_EDUCATION_TYPE (学歴)  
　DAYS_EMPLOYED (就業日数)  
　OCCUPATION_TYPE (職種)  
　BU_AMT_CREDIT_SUM_DEBT_mean (現在の負債残高平均)  


EDA、FEでは基本的事項に加え、上記の有力とされる指標から優先的に実施を行う。また、第2指標までは組合せ等も検討していきたい。  


In [ ]:
#10-1　出力　parquet(zstd形式)
# zstd圧縮で保存
'''
EDAで使用するため、データ本体からの削除はFE後に再度試走して不要カラムの削除を行う。
'''
application_train.to_parquet('application_train_cleaning.zstd.parquet', compression='zstd', index=False)
application_test.to_parquet('application_test_cleaning.zstd.parquet', compression='zstd', index=False)

import os
train_size_zstd = os.path.getsize('application_train_cleaning.zstd.parquet') / 1024**2
print(f"Trainファイルサイズ (zstd): {train_size_zstd:.2f} MB")

Trainファイルサイズ (zstd): 214.56 MB
